In [ ]:
import pandas as pd

# === 1. Load Data ===
df_ihsg = pd.read_csv("DATA_SAHAM_22-24.csv")

In [ ]:
df_ihsg["tanggal"] = pd.to_datetime(df_ihsg["tanggal"], errors="coerce")

# 2. Bersihkan kolom numerik: hapus titik ribuan, ubah ke float
num_cols = ["Terakhir", "Pembukaan", "Tertinggi", "Terendah"]
for col in num_cols:
    df_ihsg[col] = df_ihsg[col].astype(str).str.replace(".", "", regex=False)
    df_ihsg[col] = pd.to_numeric(df_ihsg[col], errors="coerce")


In [ ]:
# 3. Bersihkan kolom Volume (Vol.)
# Contoh isi: "18.26B", "12.5M", dsb.
def convert_volume(x):
    if isinstance(x, str):
        x = x.replace(",", "").strip()
        if x.endswith("B"):  # Billion
            return float(x[:-1]) * 1e9
        elif x.endswith("M"):  # Million
            return float(x[:-1]) * 1e6
        elif x.endswith("K"):  # Thousand
            return float(x[:-1]) * 1e3
        else:
            return pd.to_numeric(x, errors="coerce")
    return x

df_ihsg["Vol."] = df_ihsg["Vol."].apply(convert_volume)

In [ ]:
# 4. Bersihkan kolom Perubahan% -> float
df_ihsg["Perubahan%"] = df_ihsg["Perubahan%"].astype(str).str.replace("%", "", regex=False)
df_ihsg["Perubahan%"] = pd.to_numeric(df_ihsg["Perubahan%"], errors="coerce")

# Cek hasil bersih
print(df_ihsg.dtypes)
print(df_ihsg.head())

tanggal       datetime64[ns]
Terakhir             float64
Pembukaan            float64
Tertinggi            float64
Terendah             float64
Vol.                 float64
Perubahan%           float64
dtype: object
     tanggal  Terakhir  Pembukaan  Tertinggi  Terendah          Vol.  \
0 2022-01-03  666531.0   658626.0   667720.0  658613.0  1.826000e+10   
1 2022-01-04  669537.0   667513.0   672066.0  667513.0  1.859000e+10   
2 2022-01-05  666230.0   670317.0   673811.0  663484.0  1.791000e+10   
3 2022-01-06  665335.0   667485.0   667985.0  659323.0  1.800000e+10   
4 2022-01-07  670132.0   666951.0   671215.0  664771.0  1.571000e+10   

   Perubahan%  
0        1.27  
1        0.45  
2       -0.49  
3       -0.13  
4        0.72  


In [ ]:
df_ihsg.to_csv("DATA_SAHAM_22-24_CLEAN.csv", index=False)

In [ ]:
import pandas as pd

# Load data saham yang sudah dibersihkan
df_saham = pd.read_csv("DATA_SAHAM_22-24_CLEAN.csv")
df_saham["tanggal"] = pd.to_datetime(df_saham["tanggal"], errors="coerce")

# Load data sentimen yang sudah dibalance
df_sentimen = pd.read_csv("df_sentiment_saham_balanced.csv")
df_sentimen["tanggal"] = pd.to_datetime(df_sentimen["tanggal"], errors="coerce")

# --- Agregasi sentimen per tanggal ---
agg_sentimen = df_sentimen.groupby("tanggal").agg(
    avg_sentiment=("sentiment_score", "mean"),
    prop_pos=("sentiment_label", lambda x: (x=="positif").mean()),
    prop_neg=("sentiment_label", lambda x: (x=="negatif").mean()),
    max_impact=("sentiment_label", lambda x: x.value_counts().idxmax())
).reset_index()

# --- Merge ke data saham ---
df_final = pd.merge(df_saham, agg_sentimen, on="tanggal", how="left")

print("Data akhir:", df_final.shape)
df_final


Data akhir: (725, 11)


,tanggal,Terakhir,Pembukaan,Tertinggi,Terendah,Vol.,Perubahan%,avg_sentiment,prop_pos,prop_neg,max_impact
0,2022-01-03,666531.0,658626.0,667720.0,658613.0,1.826000e+10,1.27,NaN,NaN,NaN,NaN
1,2022-01-04,669537.0,667513.0,672066.0,667513.0,1.859000e+10,0.45,NaN,NaN,NaN,NaN
2,2022-01-05,666230.0,670317.0,673811.0,663484.0,1.791000e+10,-0.49,0.959284,0.230769,0.615385,negatif
3,2022-01-06,665335.0,667485.0,667985.0,659323.0,1.800000e+10,-0.13,0.912079,0.266667,0.400000,negatif
4,2022-01-07,670132.0,666951.0,671215.0,664771.0,1.571000e+10,0.72,0.777768,0.083333,0.666667,negatif
...,...,...,...,...,...,...,...,...,...,...,...
720,2024-12-20,698387.0,698017.0,703240.0,693158.0,1.457000e+10,0.09,0.944969,0.166667,0.000000,netral
721,2024-12-23,709644.0,703753.0,709644.0,703573.0,1.387000e+10,1.61,0.909078,0.400000,0.200000,netral
722,2024-12-24,706575.0,711564.0,712058.0,706375.0,1.106000e+10,-0.43,0.900792,0.142857,0.428571,netral
723,2024-12-27,703657.0,707338.0,710027.0,702471.0,1.443000e+10,-0.41,0.972620,0.222222,0.555556,negatif


In [ ]:
df_final.max_impact.value_counts()

,count
max_impact,
netral,282
negatif,259
positif,175


In [ ]:
# Cek semua baris yang masih ada NaN di kolom sentimen
nan_rows = df_final[df_final[["avg_sentiment", "prop_pos", "prop_neg", "max_impact"]].isna().any(axis=1)]

print("Jumlah baris dengan NaN:", len(nan_rows))
print(nan_rows[["tanggal", "avg_sentiment", "prop_pos", "prop_neg", "max_impact"]].head(20))

Jumlah baris dengan NaN: 9
       tanggal  avg_sentiment  prop_pos  prop_neg max_impact
0   2022-01-03            NaN       NaN       NaN        NaN
1   2022-01-04            NaN       NaN       NaN        NaN
95  2022-06-02            NaN       NaN       NaN        NaN
96  2022-06-03            NaN       NaN       NaN        NaN
97  2022-06-06            NaN       NaN       NaN        NaN
241        NaT            NaN       NaN       NaN        NaN
258 2023-01-17            NaN       NaN       NaN        NaN
259 2023-01-18            NaN       NaN       NaN        NaN
268 2023-02-01            NaN       NaN       NaN        NaN


In [ ]:
# Simpan hasil gabungan
df_final.to_csv("ihsg_sentimen_features.csv", index=False)

In [ ]:
df_final=pd.read_csv("ihsg_sentimen_features.csv")

In [ ]:
def categorize_sentiment(score, pos_thresh=0.6, neg_thresh=0.4):
    if pd.isna(score):
        return "tidak ada data"
    elif score >= pos_thresh:
        return "positif"
    elif score <= neg_thresh:
        return "negatif"
    else:
        return "netral"

In [ ]:
df_final["avg_sentiment_result"] = df_final["avg_sentiment"].apply(categorize_sentiment)


In [ ]:
print(df_final[["tanggal", "avg_sentiment", "avg_sentiment_result", "max_impact"]])

        tanggal  avg_sentiment avg_sentiment_result max_impact
0    2022-01-03            NaN       tidak ada data        NaN
1    2022-01-04            NaN       tidak ada data        NaN
2    2022-01-05       0.959284              positif    negatif
3    2022-01-06       0.912079              positif    negatif
4    2022-01-07       0.777768              positif    negatif
..          ...            ...                  ...        ...
720  2024-12-20       0.944969              positif     netral
721  2024-12-23       0.909078              positif     netral
722  2024-12-24       0.900792              positif     netral
723  2024-12-27       0.972620              positif    negatif
724  2024-12-30       0.922293              positif    positif

[725 rows x 4 columns]


In [ ]:
df_final

,tanggal,Terakhir,Pembukaan,Tertinggi,Terendah,Vol.,Perubahan%,avg_sentiment,prop_pos,prop_neg,max_impact,avg_sentiment_result
0,2022-01-03,666531.0,658626.0,667720.0,658613.0,1.826000e+10,1.27,NaN,NaN,NaN,NaN,tidak ada data
1,2022-01-04,669537.0,667513.0,672066.0,667513.0,1.859000e+10,0.45,NaN,NaN,NaN,NaN,tidak ada data
2,2022-01-05,666230.0,670317.0,673811.0,663484.0,1.791000e+10,-0.49,0.959284,0.230769,0.615385,negatif,positif
3,2022-01-06,665335.0,667485.0,667985.0,659323.0,1.800000e+10,-0.13,0.912079,0.266667,0.400000,negatif,positif
4,2022-01-07,670132.0,666951.0,671215.0,664771.0,1.571000e+10,0.72,0.777768,0.083333,0.666667,negatif,positif
...,...,...,...,...,...,...,...,...,...,...,...,...
720,2024-12-20,698387.0,698017.0,703240.0,693158.0,1.457000e+10,0.09,0.944969,0.166667,0.000000,netral,positif
721,2024-12-23,709644.0,703753.0,709644.0,703573.0,1.387000e+10,1.61,0.909078,0.400000,0.200000,netral,positif
722,2024-12-24,706575.0,711564.0,712058.0,706375.0,1.106000e+10,-0.43,0.900792,0.142857,0.428571,netral,positif
723,2024-12-27,703657.0,707338.0,710027.0,702471.0,1.443000e+10,-0.41,0.972620,0.222222,0.555556,negatif,positif


In [ ]:
df_final.to_csv("ihsg_sentimen_features.csv")